# Picking a model when you can't see the loss surface

In notebook 2 we fit a *cubic* to noisy cubic data. That was cheating: we **knew** the true function family, so picking the model was trivial. We could also still draw the loss surface in two notebooks ago — two parameters lived comfortably in 3-D.

In the real world, neither of those luxuries survives:

- The true function is **unknown**. All you ever see is a table of `(x, y)` observations.
- Any interesting model has **more than two parameters**, so the loss landscape lives in a space you cannot visualize.

So how do you choose a model? You **guess** — make a flexible-enough assumption, fit it, see what happens, revise. Every practitioner does this. This notebook walks through the failure mode of that guess-and-fit loop, and the standard tool we use to fix it: **regularization**.

The plan:

1. Look at the same dataset from notebook 2, but as a raw **table of observations** — no overlaid ground truth.
2. Pretend we don't know it's a cubic. Make the assumption *"a polynomial of degree 7 should be flexible enough"*.
3. Fit it with Adam, look at the result.
4. Diagnose: the model **overfits** — it bends through noise instead of capturing signal.
5. Introduce **weight decay** as the simplest form of regularization and re-fit.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import torch
import torch.nn as nn

pio.renderers.default = "notebook"

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
print(f"PyTorch: {torch.__version__}")
print(f"Device:  {device}")

N_TRAIN=10

## The observations

Same data-generating process as notebook 2 — a cubic plus Gaussian noise — but for this notebook **pretend you've never seen the line `y_clean = a_true·x³ + b_true·x² + c_true·x + d_true`**. All you have is the table below. Stare at it: you can probably tell *something* is curving, but you can't tell at a glance whether the underlying function is quadratic, cubic, exponential, or something else.

To make the choice of model harder (and overfitting easier to see), we'll use **only a small training subset** of these observations — 8 randomly chosen points — and hold out the rest as evaluation. We also extend the input range to $x \in [-4, 4]$ so the data shows the cubic both diving down on the left and climbing back up on the right; that gives the polynomial model nowhere to hide its bad extrapolation.

In [ ]:
rng = np.random.default_rng(seed=0)

# Cubic + Gaussian noise. Extend the range to ±4 so we see the cubic going
# clearly up on the right side. Double the noise relative to notebook 2 — this
# notebook is specifically about overfitting, so we want the data points to
# scatter loudly around the underlying curve.
a_true, b_true, c_true, d_true = 0.5, -1.2, -0.7, 1.0
sigma_noise = 3.0

n_samples = 60
x_np = np.linspace(-4.0, 4.0, n_samples)
y_clean = a_true * x_np**3 + b_true * x_np**2 + c_true * x_np + d_true
y_np    = y_clean + rng.normal(0.0, sigma_noise, size=n_samples)

# Small training subset; the rest is held out.
n_train = N_TRAIN
train_idx = np.sort(rng.choice(n_samples, size=n_train, replace=False))
val_idx   = np.array([i for i in range(n_samples) if i not in set(train_idx)])

x_train, y_train = x_np[train_idx], y_np[train_idx]
x_val,   y_val   = x_np[val_idx],   y_np[val_idx]

# Display the training observations as a clean DataFrame
obs = pd.DataFrame({"i": train_idx, "x": x_train.round(3), "y": y_train.round(3)})
print(f"{n_train} training observations (out of {n_samples} total), σ_noise = {sigma_noise}:")
obs.reset_index(drop=True)

In [ ]:
# Scatter the data (without overlaying the true cubic — we're pretending we don't know it).
scatter_fig = go.Figure()
scatter_fig.add_trace(go.Scatter(
    x=x_train, y=y_train, mode="markers", name=f"training (n={n_train})",
    marker=dict(size=10, color="dodgerblue", line=dict(color="black", width=1)),
))
scatter_fig.add_trace(go.Scatter(
    x=x_val, y=y_val, mode="markers", name="held out",
    marker=dict(size=6, color="lightgray", opacity=0.7),
))
scatter_fig.update_layout(
    title=f"What's the underlying function? ({n_train} training points in blue, {len(x_val)} held out in gray)",
    xaxis_title="x", yaxis_title="y",
    template="plotly_white", width=800, height=440,
)
scatter_fig.show()

## The modeling assumption: polynomial of degree 8

We don't know the underlying function. We **can't** plot the loss landscape of any reasonable model to look for its minimum — even a 4-parameter cubic puts us in 5-D, and anything richer goes further out of reach. We have to commit to an assumption and see how it fares.

Let's pick a flexible default: a **polynomial of degree 8**.

$$f(x;\, \boldsymbol{\theta}) \;=\; \theta_0 \;+\; \theta_1\, x \;+\; \theta_2\, x^2 \;+\; \theta_3\, x^3 \;+\; \theta_4\, x^4 \;+\; \theta_5\, x^5 \;+\; \theta_6\, x^6 \;+\; \theta_7\, x^7 \;+\; \theta_8\, x^8$$

Nine parameters. The reasoning is *"surely 9 parameters can capture whatever curve is hiding in there"* — and that's true, but it's also the trap. With only 8 training points, a 9-parameter polynomial has more knobs than data; it can pass arbitrarily close to *every* training point — including the noise on top of each one. That's overfitting.

To make the numerics stable, we'll normalize the input by dividing by `x_max = 4.0` before raising to powers (so the largest input magnitudes are around 1.0 instead of 4⁸ ≈ 65000). It doesn't change the function family at all, only the scale of the learned coefficients.

In [ ]:
X_MAX = 4.5  # max |x| in the data, used to scale powers down to roughly [-1, 1]

class Polynomial(nn.Module):
    """f(x) = sum_k theta_k * (x / X_MAX)^k for k in 0..degree."""
    def __init__(self, degree=N_TRAIN):
        super().__init__()
        self.degree = degree
        self.coeffs = nn.Parameter(torch.zeros(degree + 1))

    def forward(self, x):
        xn = x / X_MAX                                           # (B,)
        powers = torch.stack([xn**k for k in range(self.degree + 1)], dim=-1)  # (B, degree+1)
        return powers @ self.coeffs                              # (B,)


# Move training data onto the device
x_train_t = torch.tensor(x_train, dtype=torch.float32, device=device)
y_train_t = torch.tensor(y_train, dtype=torch.float32, device=device)
x_val_t   = torch.tensor(x_val,   dtype=torch.float32, device=device)
y_val_t   = torch.tensor(y_val,   dtype=torch.float32, device=device)


def train_polynomial(weight_decay=0.0, n_iters=200, degree=8, seed=0):
    """Train a polynomial with LBFGS (+ optional L2 weight decay).

    Why LBFGS instead of Adam here? With degree=8 the features (1, x, x², …, x⁸)
    are highly correlated, which makes the MSE loss surface very
    ill-conditioned. Adam — built for stochastic mini-batch training of large
    networks — needs many tens of thousands of steps to converge on a problem
    like this. LBFGS is a quasi-Newton optimizer that uses curvature
    information; on a small full-batch problem it reaches machine-precision
    interpolation in ~100 iterations. Same nn.Module, same loss; just a
    different optimizer.
    """
    torch.manual_seed(seed)
    model = Polynomial(degree=degree).to(device)
    optimizer = torch.optim.LBFGS(
        model.parameters(),
        lr=1.0, max_iter=20, history_size=50,
        tolerance_grad=1e-10, tolerance_change=1e-12,
        line_search_fn="strong_wolfe",
    )
    loss_fn = nn.MSELoss()

    train_losses, val_losses = [], []
    for _ in range(n_iters):
        def closure():
            optimizer.zero_grad()
            pred = model(x_train_t)
            loss = loss_fn(pred, y_train_t)
            if weight_decay > 0:
                l2 = sum((p ** 2).sum() for p in model.parameters())
                loss = loss + weight_decay * l2
            loss.backward()
            return loss
        optimizer.step(closure)
        with torch.no_grad():
            train_losses.append(loss_fn(model(x_train_t), y_train_t).item())
            val_losses.append(loss_fn(model(x_val_t),   y_val_t).item())

    return model, np.array(train_losses), np.array(val_losses)

## Fit it — no regularization yet

Standard PyTorch training loop, MSE loss. We track both the **training loss** (on our 8 points) and the **validation loss** (on the 52 held-out points). The contrast between the two is what diagnoses overfitting.

(Implementation note: see the docstring above — for this small full-batch polynomial fit we use the **LBFGS** optimizer instead of Adam, so the unregularized polynomial *actually interpolates* the training points. With 9 parameters and only 8 data points, the train MSE drops essentially to zero.)

In [ ]:
model_nowd, train_losses_nowd, val_losses_nowd = train_polynomial(weight_decay=0.0)

print(f"Final train loss = {train_losses_nowd[-1]:.4f}")
print(f"Final val   loss = {val_losses_nowd[-1]:.4f}   ← much bigger? that's overfitting")
print()
print(f"Learned coefficients (θ_0 … θ_{model_nowd.degree}):")
for k, theta in enumerate(model_nowd.coeffs.detach().cpu().numpy()):
    print(f"  θ_{k}  = {theta:+.3f}")

In [ ]:
x_dense = np.linspace(-4.5, 4.5, 400)  # slightly past training range, to expose edge wiggles
y_dense_true = a_true * x_dense**3 + b_true * x_dense**2 + c_true * x_dense + d_true

with torch.no_grad():
    y_dense_fit_nowd = model_nowd(torch.tensor(x_dense, dtype=torch.float32, device=device)).cpu().numpy()

# Lock the axes to the observed-data ranges so toggling lines on/off via the
# legend doesn't rescale the plot.
x_range = [-4.5, 4.5]
y_range = [float(y_np.min()) - 3.0, float(y_np.max()) + 3.0]

fit_fig = go.Figure()
fit_fig.add_trace(go.Scatter(x=x_val, y=y_val, mode="markers", name="held out",
                             marker=dict(size=6, color="lightgray", opacity=0.7)))
fit_fig.add_trace(go.Scatter(x=x_train, y=y_train, mode="markers", name=f"training (n={n_train})",
                             marker=dict(size=10, color="dodgerblue",
                                         line=dict(color="black", width=1))))
# Lines hidden by default — toggle from the legend during the presentation.
fit_fig.add_trace(go.Scatter(x=x_dense, y=y_dense_true, mode="lines",
                             name="true cubic (the secret)",
                             line=dict(color="crimson", width=2, dash="dash"),
                             visible="legendonly"))
fit_fig.add_trace(go.Scatter(x=x_dense, y=y_dense_fit_nowd, mode="lines",
                             name="poly-8 fit (no regularization)",
                             line=dict(color="orange", width=3),
                             visible="legendonly"))
fit_fig.update_layout(
    title=f"Polynomial-of-degree-8 fit to {n_train} training points",
    xaxis_title="x", yaxis_title="y",
    xaxis=dict(range=x_range, autorange=False),
    yaxis=dict(range=y_range, autorange=False),
    template="plotly_white", width=850, height=460,
)
fit_fig.show()

# Loss curves
loss_fig = go.Figure()
loss_fig.add_trace(go.Scatter(y=train_losses_nowd, mode="lines",
                              name="train MSE", line=dict(color="dodgerblue", width=2)))
loss_fig.add_trace(go.Scatter(y=val_losses_nowd, mode="lines",
                              name="val MSE", line=dict(color="crimson", width=2)))
loss_fig.update_layout(
    title="Training vs validation loss (no regularization)",
    xaxis_title="step", yaxis_title="MSE",
    yaxis_type="log",
    template="plotly_white", width=850, height=360,
)
loss_fig.show()

## What you should see — and what it means

Two signs of **overfitting**:

1. **The orange curve bends through every blue training point** but takes off wildly on either edge of the training range. That's the polynomial spending its degree-8 flexibility on memorizing the noise in those 8 specific samples, not on capturing the underlying smooth curve.
2. **The validation loss is much higher than the training loss** — the gap between the blue (train) and red (val) curves on the loss plot. The model nailed the training data and is bad at generalizing.

Look at the coefficient magnitudes printed above. They're often dozens or hundreds in absolute value. That's the signature of overfitting in polynomial regression: tiny changes in the data direction get amplified into huge swings, achieved by piling up large coefficients with opposing signs that nearly cancel. The model has the right *fit*, but not the right *shape*.

The fundamental tension is:

- We don't *want* less flexibility — the model needs to be flexible enough to capture whatever the true function is.
- We *do* want a way to tell the optimizer **"prefer simpler solutions when several explain the data equally well"**.

That's exactly what **regularization** does.

## Regularization: weight decay (L2)

The simplest form is **L2 regularization**, also called **weight decay**. We add a penalty term proportional to the squared norm of the parameters to the loss:

$$\mathcal{L}_\text{reg}(\boldsymbol{\theta}) \;=\; \underbrace{\text{MSE}(y, \hat{y})}_{\text{data fit}} \;+\; \lambda \cdot \underbrace{\sum_k \theta_k^2}_{\text{penalty on big weights}}$$

Now the optimizer faces a tradeoff: it can keep shrinking the data-fit term by adding extra polynomial flexibility, but every time it does, the penalty term grows. So it learns to use as little parameter magnitude as possible to explain the data — which means **simpler-shaped fits**.

In PyTorch this is one extra kwarg to the optimizer:

```python
torch.optim.Adam(model.parameters(), lr=0.02, weight_decay=λ)
```

That's it. Same training loop, same model, same data. Just one number tuned. Let's try $\lambda = 0.05$.

In [ ]:
LAMBDA = 0.05

model_wd, train_losses_wd, val_losses_wd = train_polynomial(weight_decay=LAMBDA)

print(f"Final train loss (no wd) = {train_losses_nowd[-1]:.4f}    val loss = {val_losses_nowd[-1]:.4f}")
print(f"Final train loss (wd={LAMBDA}) = {train_losses_wd[-1]:.4f}    val loss = {val_losses_wd[-1]:.4f}")
print()
print("Learned coefficients with weight decay:")
for k, theta in enumerate(model_wd.coeffs.detach().cpu().numpy()):
    print(f"  θ_{k}  = {theta:+.3f}")

In [ ]:
with torch.no_grad():
    y_dense_fit_wd = model_wd(torch.tensor(x_dense, dtype=torch.float32, device=device)).cpu().numpy()

# Reuse the locked axis ranges from the previous plot so toggles don't rescale.
compare_fig = go.Figure()
compare_fig.add_trace(go.Scatter(x=x_val, y=y_val, mode="markers", name="held out",
                                 marker=dict(size=6, color="lightgray", opacity=0.7)))
compare_fig.add_trace(go.Scatter(x=x_train, y=y_train, mode="markers", name=f"training (n={n_train})",
                                 marker=dict(size=10, color="dodgerblue",
                                             line=dict(color="black", width=1))))
# Lines hidden by default; toggle them on from the legend.
compare_fig.add_trace(go.Scatter(x=x_dense, y=y_dense_true, mode="lines",
                                 name="true cubic",
                                 line=dict(color="crimson", width=2, dash="dash"),
                                 visible="legendonly"))
compare_fig.add_trace(go.Scatter(x=x_dense, y=y_dense_fit_nowd, mode="lines",
                                 name="poly-8, λ = 0 (overfit)",
                                 line=dict(color="orange", width=2),
                                 visible="legendonly"))
compare_fig.add_trace(go.Scatter(x=x_dense, y=y_dense_fit_wd, mode="lines",
                                 name=f"poly-8, λ = {LAMBDA} (regularized)",
                                 line=dict(color="seagreen", width=3),
                                 visible="legendonly"))
compare_fig.update_layout(
    title="Same model, same data — only weight decay differs",
    xaxis_title="x", yaxis_title="y",
    xaxis=dict(range=x_range, autorange=False),
    yaxis=dict(range=y_range, autorange=False),
    template="plotly_white", width=850, height=460,
)
compare_fig.show()

loss_compare = go.Figure()
loss_compare.add_trace(go.Scatter(y=val_losses_nowd, mode="lines",
                                  name="val MSE, λ = 0",
                                  line=dict(color="orange", width=2)))
loss_compare.add_trace(go.Scatter(y=val_losses_wd, mode="lines",
                                  name=f"val MSE, λ = {LAMBDA}",
                                  line=dict(color="seagreen", width=2)))
loss_compare.update_layout(
    title="Validation loss: regularization keeps the generalization gap closed",
    xaxis_title="step", yaxis_title="validation MSE",
    yaxis_type="log",
    template="plotly_white", width=850, height=360,
)
loss_compare.show()

## Takeaways

- **Without ground truth or a visualizable loss surface, model choice is a guess.** You pick something flexible, fit it, and revise based on what you observe — especially the gap between training and validation loss.
- **More parameters than data points = potential overfitting.** The model has enough capacity to memorize noise, and it will if you let it.
- **Regularization is the standard fix.** Weight decay (L2) adds a penalty for large parameter values to the loss, biasing the optimizer toward simpler solutions. In PyTorch it's a single optimizer kwarg.
- **It generalizes far beyond polynomials.** Every neural network you train from here on — including the MLP, the RNN in this series, and GPT itself — uses weight decay (or a related technique like dropout, gradient clipping, or label smoothing) to fight overfitting. The principle is identical: shape the loss so the optimizer prefers "simpler" models when several explain the data equally well.

Things to try yourself:

- Bump `LAMBDA` to `1.0` or `5.0`. What happens to the fit? (Spoiler: **underfitting** — the penalty becomes louder than the data.)
- Drop `LAMBDA` to `1e-4`. Is overfitting back?
- Increase `n_train` from `8` to `40`. Does the unregularized polynomial still overfit, or does the extra data alone fix it?
- Change `degree=8` to `degree=15` in the no-regularization run. Even with 8 training points and `weight_decay=0`, can you get a smooth fit by adjusting `lr` and step count alone? (No — without regularization, you eventually overfit; more capacity just makes it faster.)